## Map SIF network nodes (UniProt and ChEBI identifiers) to Gene Names (for proteins) and Molecule Names (for molecules)

In [ ]:
from SPARQLWrapper import SPARQLWrapper, TURTLE, JSON, CSV
import subprocess
import time
import os
from requests.utils import requote_uri
from urllib.parse import quote
import re
import rdflib
import pandas as pd

In [ ]:
results_files = list()
for file in os.listdir("../../Results/ReactomeHomoSapiens95/Meaning2/QueryResults/"):
    print(file)
    results_files.append(file)

print(results_files)
aggregatedNetwork = pd.concat([pd.read_csv(f"../../Results/ReactomeHomoSapiens95/Meaning2/QueryResults/{file}", header=0, sep=",") for file in results_files]) 
print(aggregatedNetwork.head()) 
print(len(aggregatedNetwork))

In [ ]:
# get back mapping of Reactome identifiers
Reactome95EntitiesIDS = pd.read_csv("../../Results/ReactomeHomoSapiens95/UtilityFiles/ReactomeHomoSapiens95EntityRefsIDs.csv", sep=",", header=0)

dico_entity_ref = dict()
dico_entity_ref_names = dict()
for index, row in Reactome95EntitiesIDS.iterrows():
    ref = row[0].replace("http://www.reactome.org/biopax/95/48887#", "reactome:")
    if not ref in dico_entity_ref.keys():
        dico_entity_ref[ref] = row[2]
        dico_entity_ref_names[ref] = row[1]

print(dico_entity_ref)
print(dico_entity_ref_names)
print(len(dico_entity_ref))

In [ ]:
aggregatedNetworkIDS = pd.DataFrame(columns=["Source", "Interaction", "Target"])
i = 0
for index, row in aggregatedNetwork.iterrows():
    entity1 = row[0]
    entity2 = row[2]
    interaction = row[1]
    entity1ID = dico_entity_ref[entity1]
    entity2ID = dico_entity_ref[entity2]
    aggregatedNetworkIDS.at[i, "Source"] = entity1ID
    aggregatedNetworkIDS.at[i, "Interaction"] = interaction
    aggregatedNetworkIDS.at[i, "Target"] = entity2ID
    i += 1

print(aggregatedNetworkIDS.head())
print(len(aggregatedNetworkIDS))
aggregatedNetworkIDS.to_csv("../../Results/ReactomeHomoSapiens95/Meaning2/SIF/Reactome95_SIF_Meaning2.tsv", sep="\t", header=1, index=None)
aggregatedNetworkIDS.to_csv("../../Results/ReactomeHomoSapiens95/Meaning2/SIF/Reactome95_SIF_Meaning2.sif", sep=" ", header=1, index=None)

In [ ]:
# create node table SIF
SIF_entities = pd.DataFrame(columns=["ID"])
entity_list = list()
for index, row in aggregatedNetworkIDS.iterrows():
    if not row[0] in entity_list:
        entity_list.append(row[0])
    if not row[2] in entity_list:
        entity_list.append(row[2])

print(entity_list)

SIF_entities["ID"] = entity_list
SIF_entities.to_csv("../../Results/ReactomeHomoSapiens95/UtilityFiles/Entities-Reactome95-SIF-Meaning2.txt", header=None, index=False)
print(SIF_entities.head())

### Map Uniprot IDs to Gene Names

1. Create SIF node table with proteins represented with their uniprot ids
2. Go to https://www.uniprot.org/help/id_mapping and load SIF node table as text file and run mapping "UniProtKB AC/ID" to "UniProtKB/Swiss-Prot" (get back the mappings of the UniProt IDs to UniProt reviewed IDs associated to a unique Gene Name) + filter Homo Sapiens results
3. Download results as tsv file "../../Data/UniprotMapping/Reactome95-Meaning2-SIFProteinsReviewed.tsv"
4. Create dictionary of Uniprot IDs and Gene names

In [ ]:
# load uniprot mapping file and sotre data in a dict
uniprot_mapping_sif_file = pd.read_table("../../Data/Uniprot_mapping/Reactome95-Meaning2-SIFProteinsReviewed.tsv", sep="\t")
print(uniprot_mapping_sif_file.head())
dico_SIF_uniprot_mapping = dict()
for item, row in uniprot_mapping_sif_file.iterrows():
    if row[0] not in dico_SIF_uniprot_mapping.keys():
        dico_SIF_uniprot_mapping[row[0]] = row[5]

print(dico_SIF_uniprot_mapping)

### Map ChEBI IDs to molecule names

1. get back mapping file from https://www.ebi.ac.uk/chebi/downloads
2. Create dictionary of ChEBI identifiers and associated molecule names

In [ ]:
chebi_mapping_file = pd.read_table("../../Data/Chebi_mapping/names.tsv")
print(chebi_mapping_file.head())

In [ ]:
dico_SIF_chebi_mapping = dict()
for index, row in chebi_mapping_file.iterrows():
    chebi_id = f"CHEBI:{row[1]}"
    chebi_name = row[7]
    if not chebi_id in dico_SIF_chebi_mapping.keys():
        dico_SIF_chebi_mapping[chebi_id] = row[7]
print(dico_SIF_chebi_mapping)

### Create new SIF network with Gene names and molecule names


In [ ]:
aggregatedNetworkGeneNames = pd.DataFrame(columns=["Source", "Interaction", "Target"])
i = 0
unmapped_entities = list()
for index, row in aggregatedNetworkIDS.iterrows():
    entity1 = row[0]
    entity2 = row[2]
    if entity1 in dico_SIF_uniprot_mapping.keys():
        entity1Name = dico_SIF_uniprot_mapping[entity1]
    elif entity1 in dico_SIF_chebi_mapping.keys():
        entity1Name = dico_SIF_chebi_mapping[entity1]
    else:
        entity1Name = entity1
        unmapped_entities.append(entity1)
    aggregatedNetworkGeneNames.at[i, "Source"] = entity1Name
    aggregatedNetworkGeneNames.at[i, "Interaction"] = row[1]
    if entity2 in dico_SIF_uniprot_mapping.keys():
        entity2Name = dico_SIF_uniprot_mapping[entity2]
    elif entity2 in dico_SIF_chebi_mapping.keys():
        entity2Name = dico_SIF_chebi_mapping[entity2]
    else:
        entity2Name = entity2
        unmapped_entities.append(entity2)
    aggregatedNetworkGeneNames.at[i, "Target"] = entity2Name
    i += 1

print(unmapped_entities)
aggregatedNetworkGeneNames.to_csv("../../Results/ReactomeHomoSapiens95/Meaning2/SIF/Reactome95-SIF-Meaning2-Names.tsv", sep="\t", header=0, index=False)